# Avaliação Acadêmica - Selective Risk Framework
Esse notebook extrai os dados das execuções agregadas (`outputs/final_eval/`) e gera gráficos no padrão acadêmico 

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import seaborn as sns

plt.style.use('seaborn-v0_8-paper')
sns.set_theme(style="whitegrid", context="paper", font_scale=1.3)

BASE_DIR = '../outputs/final_eval'
CORPUS = 'IntentPTCorpus' # Mude para o dataset desejado (ex: Banking77Corpus)
KSHOT = 'kshot_5'
LANG_DIR = 'br' # Use 'en' para datasets em inglês

# Métodos que entrarão no gráfico
METHODS = [
    ("MSP", "baseline", "msp"),
    ("Energy", "baseline", "energy"),
    ("Mahalanobis", "baseline", "mahalanobis"),
    ("kNN", "baseline", "knn"),
    ("ConjNorm", "baseline", "sota_conjnorm"),
    ("LAQDA", "laqda", ""),
    ("LAQDA+SGR", "laqda_sgr", ""),
    ("LAQDA+SGR (X-Maha)", "laqda_sgr", "xmaha")
]

In [ ]:
def load_aggregated_data(base_path, lang_dir, folder, corpus, kshot, suffix):
    folds = [f"fold_0{i}" for i in range(1, 6)]
    metrics = {"accuracy": [], "aurc": [], "auroc": [], "sgr_coverage_at_risk_10": [], "fpr_at_95": []}
    
    for fold in folds:
        fname = "test_final_metrics_report.json" if suffix == "" else f"test_final_{suffix}_metrics_report.json"
        path = os.path.join(base_path, lang_dir, folder, corpus, fold, kshot, fname)
        if os.path.exists(path):
            with open(path) as f:
                d = json.load(f)
                for k in metrics.keys():
                    if k in d and d[k] is not None:
                        metrics[k].append(d[k])
    
    if not metrics['accuracy']:
        return None
        
    return {k: (np.mean(v), np.std(v)) for k, v in metrics.items() if v}

data_rows = []
for label, folder, suffix in METHODS:
    res = load_aggregated_data(BASE_DIR, LANG_DIR, folder, CORPUS, KSHOT, suffix)
    if res:
        data_rows.append({
            "Metodo": label,
            "ACC_mean": res.get('accuracy', (0,0))[0], "ACC_std": res.get('accuracy', (0,0))[1],
            "AURC_mean": res.get('aurc', (0,0))[0], "AURC_std": res.get('aurc', (0,0))[1],
            "AUROC_mean": res.get('auroc', (0,0))[0], "AUROC_std": res.get('auroc', (0,0))[1],
            "SGR10_mean": res.get('sgr_coverage_at_risk_10', (0,0))[0], "SGR10_std": res.get('sgr_coverage_at_risk_10', (0,0))[1],
            "FPR95_mean": res.get('fpr_at_95', (0,0))[0], "FPR95_std": res.get('fpr_at_95', (0,0))[1]
        })

df = pd.DataFrame(data_rows)
print(f"Dados carregados: {len(df)} métodos.")
display(df)

### Gráfico 1: Trade-off de Risco vs Cobertura (AURC vs SGR@10%)
Esse gráfico clássico de barras duplas demonstra como os métodos se saem ao equilibrar cobertura de dados e risco. Ideal para provar a eficácia do SGR.

In [ ]:
if not df.empty:
    fig, ax1 = plt.subplots(figsize=(10, 5))
    
    x = np.arange(len(df))
    width = 0.35
    
    color_aurc = '#d9534f' # Vermelho elegante
    ax1.bar(x - width/2, df['AURC_mean'], width, yerr=df['AURC_std'], 
            label='AURC (↓ Menor é Melhor)', color=color_aurc, alpha=0.8, capsize=4)
    ax1.set_ylabel('AURC', color=color_aurc, fontweight='bold')
    ax1.tick_params(axis='y', labelcolor=color_aurc)
    
    ax2 = ax1.twinx()
    color_sgr = '#5bc0de' # Azul limpo
    ax2.bar(x + width/2, df['SGR10_mean'], width, yerr=df['SGR10_std'], 
            label='SGR Coverage @ 10% (↑ Maior é Melhor)', color=color_sgr, alpha=0.8, capsize=4)
    ax2.set_ylabel('SGR Coverage @ 10%', color=color_sgr, fontweight='bold')
    ax2.tick_params(axis='y', labelcolor=color_sgr)
    
    ax1.set_xticks(x)
    ax1.set_xticklabels(df['Metodo'], rotation=30, ha='right', fontweight='bold')
    
    plt.title(f'Risco vs Cobertura - {CORPUS} ({KSHOT})', fontsize=14, fontweight='bold', pad=20)
    fig.tight_layout()
    
    # Salva em formato vetorial para o artigo
    plt.savefig('plot_1_aurc_vs_sgr.pdf', dpi=300, bbox_inches='tight')
    plt.show()

### Gráfico 2: Detecção Out-of-Distribution (AUROC vs FPR@95)
Gráfico de dispersão (scatter) comparando a capacidade dos modelos de rejeitar dados OOD enquanto tentam aceitar os ID.

In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(8, 6))
    
    for i, row in df.iterrows():
        # Destaque visual para os métodos LAQDA
        is_proposed = 'LAQDA' in row['Metodo']
        color = '#5cb85c' if is_proposed else '#f0ad4e'
        marker = 'D' if is_proposed else 'o'
        size = 12 if is_proposed else 9
        
        ax.errorbar(row['FPR95_mean'], row['AUROC_mean'], 
                    xerr=row['FPR95_std'], yerr=row['AUROC_std'], 
                    fmt=marker, markersize=size, color=color, 
                    capsize=3, alpha=0.9, label='_nolegend_')
        
        # Anotação de texto próximo ao ponto
        ax.annotate(row['Metodo'], (row['FPR95_mean'], row['AUROC_mean']),
                    xytext=(8, 5), textcoords='offset points', fontsize=10,
                    fontweight='bold' if is_proposed else 'normal')
    
    ax.set_xlabel('FPR @ 95% TPR (↓ Menor é Melhor)', fontweight='bold')
    ax.set_ylabel('AUROC (↑ Maior é Melhor)', fontweight='bold')
    ax.set_title(f'Separação OOD - {CORPUS}', fontsize=14, fontweight='bold', pad=15)
    
    # Inverte o eixo X para que o canto superior direito seja o "melhor dos mundos" (Alto AUROC, Baixo FPR)
    ax.invert_xaxis()
    ax.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.savefig('plot_2_ood_detection.pdf', dpi=300, bbox_inches='tight')
    plt.show()